# Сверка дашборда и портала (final_script_2_1)

Январь–июль 2026. Колонка **ДБ** — витрина актуального скрипта `final_df_period_2026_01_2026_08_final_script_2_1.csv`. Старый `final_script_2` не читается. Колонка **Портал** — месячные Excel `01_Январь_2026.xlsx` … `07_Июль_2026.xlsx`.

Тетрадка не пересобирает витрину и не ходит в озеро. Запуск на Jupyter, каталог `/home/jovyan/documents/Equaring/Data`.

Δ = ДБ − Портал. Знак сумм не меняется.

| Строка | ДБ | Портал |
|---|---|---|
| Всего клиентов | договоры `agr_id` | `ID договора` / `Номер договора` |
| Активные | `active_terms > 0` | колонка активных терминалов, иначе `Кол-во терминалов > 0` |
| Неактивные | остальные договоры месяца | то же |
| Приток | месяц `d_valid_from` = месяц `snapshot_month_start` | месяц даты открытия = месяц файла |
| Отток | месяц `d_valid_to` = месяц снимка, дата не пустая | месяц даты закрытия = месяц файла |
| Кол-во ТТ, терминалы, деньги | сумма после свёртки ИНН + договор (`max`, как `build_lake_agg`) | то же |
| Комиссия (% эквайринга) | `SUM(commission_from_ops) / SUM(trx_sum)` | та же доля, в процентах |

Лист `сверка_руб` — деньги в рублях. Лист `сверка_компакт` — оборот в млрд, остальные деньги в млн, счётчики и процент как есть.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
MONTHS = [f'2026-{m:02d}' for m in range(1, 8)]
MONTH_LABEL = {m: f'{m[5:7]}.{m[2:4]}' for m in MONTHS}

FINAL_DF_CSV = DATA_DIR / 'final_df_period_2026_01_2026_08_final_script_2_1.csv'
CHECKPOINT_DIR = DATA_DIR / 'checkpoints_final_df_2026_01_2026_08_final_script_2_1'
OUT_XLSX = DATA_DIR / 'portal_vs_dashboard_2026_01_2026_07_final_script_2_1.xlsx'

excel_header = 0
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
    '2026-07': -1,
}
excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
    '2026-07': DATA_DIR / '07_Июль_2026.xlsx',
}

# Комиссия с операций — раньше общей «Комиссия эквайринга»:
# в col_map final_script это имя стоит и в comm_ops, и в comm_total.
COL_MAP = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
    'term_col': ['Кол-во терминалов', 'Количество терминалов'],
    'active_terms_col': [
        'Кол-во активных терминалов', 'Количество активных терминалов',
        'Активные терминалы', 'Активных терминалов', 'active_terms',
    ],
    'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'Сумма транзакций', 'trx_sum'],
    'comm_ops_col': [
        'Комиссия (% с операций)', 'Комиссия \n(% с операций)', 'Комиссия % с операций',
        'Комиссия с операций', 'commission_from_ops',
        'Комиссия эквайринга',
    ],
    'comm_monthly_col': [
        'Комиссия в месяц', 'Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)',
    ],
    'int_component_col': [
        'Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)',
        'Комиссия МПС (IRF)', 'IRF', 'Комиссия МПС',
    ],
    'chod_col': ['ЧОД', 'chod'],
    'aur_col': ['АУР', 'AUR', 'Aur', 'Аур', 'АУР, руб', 'АУР руб'],
    'amortization_col': [
        'Амортизация', 'Аморт', 'Амортизация терминалов',
        'amortization', 'Amortization', 'Амортизация, руб',
    ],
    'fin_result_col': [
        'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
        'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
        'fin_result', 'Fin.Res.', 'FinRes',
    ],
    'open_col': [
        'Дата открытия', 'Дата открытия договора', 'Дата начала договора',
        'Дата начала', 'd_valid_from', 'Дата заключения',
    ],
    'close_col': [
        'Дата закрытия', 'Дата закрытия договора', 'Дата расторжения',
        'd_valid_to', 'Дата окончания',
    ],
}

SUM_FIELDS = [
    'retl_cnt', 'term_cnt', 'trx_sum', 'commission_from_ops', 'commission_monthly',
    'int_component', 'chod', 'aur', 'amortization', 'fin_result',
]

METRIC_ROWS = [
    ('clients_total', 'Всего клиентов', 'count'),
    ('clients_active', 'Активные клиенты', 'count'),
    ('clients_inactive', 'Неактивные клиенты', 'count'),
    ('inflow', 'Приток', 'count'),
    ('outflow', 'Отток', 'count'),
    ('retl_cnt', 'Кол-во ТТ', 'count'),
    ('term_cnt', 'Кол-во терминалов', 'count'),
    ('trx_sum', 'Сумма транзакций', 'bln'),
    ('chod', 'ЧОД', 'mln'),
    ('fin_result', 'Фин. рез.', 'mln'),
    ('aur', 'АУР', 'mln'),
    ('amortization', 'Амортизация', 'mln'),
    ('int_component', 'IRF', 'mln'),
    ('commission_monthly', 'Комиссия месяц', 'mln'),
    ('acq_pct', 'Комиссия (% эквайринга)', 'pct'),
]


def assert_script_21(path):
    if 'final_script_2_1' not in Path(path).name:
        raise RuntimeError(f'Отказ: это не файл final_script_2_1: {path}')


def month_key(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if len(s) >= 7 and s[4] == '-':
        return s[:7]
    try:
        return pd.to_datetime(s).strftime('%Y-%m')
    except Exception:
        return None


def pick_col(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').replace('\n', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    if s is None:
        return pd.Series(dtype='float64')
    if not isinstance(s, pd.Series):
        s = pd.Series(s)
    raw = s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False)
    raw = raw.str.replace(',', '.', regex=False)
    return pd.to_numeric(raw, errors='coerce')


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None', '<NULL>'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_month_label(series):
    """Месяц YYYY-MM. Пустая строка и мусор — пусто. Число Excel-serial тоже читается."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return series.dt.strftime('%Y-%m')
    raw = series.astype(str).str.strip()
    raw = raw.replace({
        '': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NULL>': pd.NA,
        'NaT': pd.NA, 'nat': pd.NA, 'NULL': pd.NA,
    })
    as_dt = pd.to_datetime(raw, errors='coerce')
    num = pd.to_numeric(raw, errors='coerce')
    serial = as_dt.isna() & num.notna() & num.between(20000, 80000)
    if serial.any():
        as_dt = as_dt.copy()
        as_dt.loc[serial] = pd.to_datetime(num.loc[serial], unit='D', origin='1899-12-30')
    return as_dt.dt.strftime('%Y-%m')


def first_valid_month(s):
    s = s.dropna()
    s = s[s.astype(str).str.strip().ne('')]
    s = s[s.astype(str) != 'NaT']
    return s.iloc[0] if len(s) else np.nan


def read_month_excel(path, header):
    if header != -1:
        return pd.read_excel(path, header=header)
    raw = pd.read_excel(path, header=None, nrows=12)
    markers = (
        'аур', 'амортизац', 'фин. рез', 'фин.рез', 'фин рез',
        'fin.res', 'fin_result', 'чод', 'инн',
    )
    for i, row in raw.iterrows():
        cells = ' '.join(
            str(x).lower().replace('\n', ' ')
            for x in row.tolist() if pd.notna(x)
        )
        if any(m in cells for m in markers):
            print(f'{path.name}: header=-1 → строка заголовка {i}')
            return pd.read_excel(path, header=i)
    print(f'{path.name}: header=-1, строка заголовка не найдена, header=None')
    return pd.read_excel(path, header=None)


def guess_date_col(columns, kind):
    """kind: open | close. Не берём дату привязки P."""
    if kind == 'open':
        keys = ('открыт', 'дата начала')
    else:
        keys = ('закрыт', 'расторж')
    hits = []
    for c in columns:
        n = re.sub(r'\s+', ' ', str(c).replace('\n', ' ').strip().lower())
        if 'привяз' in n or n.endswith(' p') or ' p ' in f' {n} ':
            continue
        if any(k in n for k in keys):
            hits.append(c)
    if not hits:
        return None
    for c in hits:
        if 'договор' in str(c).lower():
            return c
    return hits[0]


print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())
print('final_df csv exists=', FINAL_DF_CSV.exists(), FINAL_DF_CSV.name)
print('checkpoints exists=', CHECKPOINT_DIR.exists())
for k, v in excel_reference_by_month.items():
    print(f'Excel {k}: exists={Path(v).exists()} | {Path(v).name}')


In [ ]:
def load_final_df():
    if FINAL_DF_CSV.exists():
        assert_script_21(FINAL_DF_CSV)
        df = pd.read_csv(FINAL_DF_CSV, dtype=str, low_memory=False)
        return df, f'csv: {FINAL_DF_CSV.name}'

    if not CHECKPOINT_DIR.exists():
        raise RuntimeError(
            'Нет final_df final_script_2_1. Нужен '
            f'{FINAL_DF_CSV.name} или каталог {CHECKPOINT_DIR.name}. '
            'Старый final_script_2 не подставляется.'
        )
    assert_script_21(CHECKPOINT_DIR)
    month_files = sorted(CHECKPOINT_DIR.glob('final_df_2026_*.parquet')) + sorted(
        CHECKPOINT_DIR.glob('final_df_2026_*.csv')
    )
    if not month_files:
        raise RuntimeError(f'В {CHECKPOINT_DIR} нет final_df_2026_*.parquet/csv')
    frames = []
    seen = set()
    for f in month_files:
        if f.stem in seen:
            continue
        seen.add(f.stem)
        if f.suffix == '.parquet':
            frames.append(pd.read_parquet(f))
        else:
            frames.append(pd.read_csv(f, dtype=str, low_memory=False))
    return pd.concat(frames, ignore_index=True), f'checkpoints: {CHECKPOINT_DIR.name}'


final_df, source_used = load_final_df()
need = {
    'report_month', 'agr_id', 'inn', 'retl_cnt', 'term_cnt', 'active_terms',
    'trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component',
    'chod', 'aur', 'amortization', 'fin_result', 'd_valid_from', 'd_valid_to',
}
missing = need - set(final_df.columns)
if missing:
    raise RuntimeError(f'В final_df нет колонок: {sorted(missing)}')

work = final_df.copy()
work['report_month'] = work['report_month'].map(month_key)
if 'snapshot_month_start' not in work.columns:
    work['snapshot_month_start'] = work['report_month']
    print('snapshot_month_start нет — для притока/оттока берётся report_month')

present = sorted(x for x in work['report_month'].dropna().unique().tolist())
extra = [m for m in present if m not in MONTHS]
work = work[work['report_month'].isin(MONTHS)].copy()
print('source:', source_used)
print('rows Jan–Jul:', f'{len(work):,}', '| months:', sorted(work['report_month'].dropna().unique().tolist()))
if extra:
    print('вне окна, в сверку не входят:', extra)


def dedup_sums(frame, inn_key, agr_key):
    """Одна строка на ИНН + договор, max, затем сумма. Как build_lake_agg."""
    sub = frame.dropna(subset=[agr_key]).copy()
    for c in SUM_FIELDS:
        sub[c] = to_num_series(sub[c])
    if inn_key in sub.columns and sub[inn_key].notna().any():
        sub = sub.dropna(subset=[inn_key])
        keys = [inn_key, agr_key]
    else:
        keys = [agr_key]
    if sub.empty:
        return {c: np.nan for c in SUM_FIELDS}
    agg = sub.groupby(keys, as_index=False)[SUM_FIELDS].max()
    out = {}
    for c in SUM_FIELDS:
        out[c] = float(agg[c].fillna(0).sum())
    return out


def client_and_flow(frame, agr_key, active_col, snap_labels, open_labels, close_labels, active_mode):
    """active_mode: column | term_cnt. snap/open/close — Series месяца YYYY-MM, индекс как у frame."""
    sub = frame.dropna(subset=[agr_key]).copy()
    sub['_snap'] = snap_labels.reindex(sub.index)
    sub['_open'] = open_labels.reindex(sub.index)
    sub['_close'] = close_labels.reindex(sub.index)
    if active_mode == 'column' and active_col in sub.columns:
        sub['_active'] = to_num_series(sub[active_col])
    else:
        sub['_active'] = to_num_series(sub['term_cnt']) if 'term_cnt' in sub.columns else 0.0
    if sub.empty:
        return {
            'clients_total': np.nan,
            'clients_active': np.nan,
            'clients_inactive': np.nan,
            'inflow': np.nan,
            'outflow': np.nan,
        }
    per = sub.groupby(agr_key, as_index=False).agg(
        _active=('_active', 'max'),
        _snap=('_snap', first_valid_month),
        _open=('_open', first_valid_month),
        _close=('_close', first_valid_month),
    )
    n_total = int(per[agr_key].nunique())
    n_active = int(per.loc[per['_active'].fillna(0) > 0, agr_key].nunique())
    inflow = int(per.loc[per['_open'].notna() & (per['_open'] == per['_snap']), agr_key].nunique())
    outflow = int(per.loc[per['_close'].notna() & (per['_close'] == per['_snap']), agr_key].nunique())
    return {
        'clients_total': n_total,
        'clients_active': n_active,
        'clients_inactive': n_total - n_active,
        'inflow': inflow,
        'outflow': outflow,
    }


def acq_pct(sums):
    trx = sums.get('trx_sum')
    comm = sums.get('commission_from_ops')
    if trx is None or pd.isna(trx) or abs(trx) < 1e-9 or comm is None or pd.isna(comm):
        return np.nan
    return 100.0 * float(comm) / float(trx)


def metrics_from_parts(clients, sums):
    row = dict(clients)
    row.update(sums)
    row['acq_pct'] = acq_pct(sums)
    return row


rows = []
audit = []

for month in MONTHS:
    g = work.loc[work['report_month'] == month].copy()
    g['inn_key'] = g['inn'].map(normalize_inn_q1)
    g['agr_id_key'] = g['agr_id'].map(normalize_agr_q1)
    snap = to_month_label(g['snapshot_month_start'])
    opened = to_month_label(g['d_valid_from'])
    closed = to_month_label(g['d_valid_to'])
    clients = client_and_flow(g, 'agr_id_key', 'active_terms', snap, opened, closed, 'column')
    sums = dedup_sums(g, 'inn_key', 'agr_id_key')
    db = metrics_from_parts(clients, sums)
    db_status = 'ok' if len(g) else 'no_rows'

    portal = {k: np.nan for k, _label, _scale in METRIC_ROWS}
    path = Path(excel_reference_by_month[month])
    if not path.exists():
        print(f'{month}: нет файла портала {path.name}')
        audit.append({'report_month': month, 'field': 'file', 'excel_column': None, 'status': 'file_missing'})
    else:
        header = excel_header_by_month.get(month, excel_header)
        ex = read_month_excel(path, header)
        resolved = {k: pick_col(ex.columns, v) for k, v in COL_MAP.items()}
        if resolved['open_col'] is None:
            resolved['open_col'] = guess_date_col(ex.columns, 'open')
            open_status = 'guessed' if resolved['open_col'] else 'missing'
        else:
            open_status = 'exact'
        if resolved['close_col'] is None:
            resolved['close_col'] = guess_date_col(ex.columns, 'close')
            close_status = 'guessed' if resolved['close_col'] else 'missing'
        else:
            close_status = 'exact'

        if resolved['open_col'] is None or resolved['close_col'] is None:
            print(f'{month}: нет колонки даты для притока/оттока.')
            print('  открытие:', resolved['open_col'], open_status)
            print('  закрытие:', resolved['close_col'], close_status)
            print('  колонки файла:', list(ex.columns))

        if resolved['agr_col'] is None:
            print(f'{month}: нет ID договора. Колонки: {list(ex.columns)}')
            audit.append({'report_month': month, 'field': 'agr', 'excel_column': None, 'status': 'missing'})
        else:
            ex = ex.copy()
            if resolved['inn_col'] is None:
                print(f'{month}: нет ИНН, денежные суммы сворачиваются только по договору')
                ex['inn_key'] = pd.Series(pd.NA, index=ex.index)
            else:
                ex['inn_key'] = ex[resolved['inn_col']].map(normalize_inn_q1)
            ex['agr_id_key'] = ex[resolved['agr_col']].map(normalize_agr_q1)
            for field, col_key in [
                ('retl_cnt', 'retl_col'),
                ('term_cnt', 'term_col'),
                ('trx_sum', 'trx_sum_col'),
                ('commission_from_ops', 'comm_ops_col'),
                ('commission_monthly', 'comm_monthly_col'),
                ('int_component', 'int_component_col'),
                ('chod', 'chod_col'),
                ('aur', 'aur_col'),
                ('amortization', 'amortization_col'),
                ('fin_result', 'fin_result_col'),
            ]:
                col = resolved[col_key]
                ex[field] = to_num_series(ex[col]) if col else np.nan
                audit.append({
                    'report_month': month,
                    'field': field,
                    'excel_column': col,
                    'status': 'ok' if col else 'missing',
                })

            active_col_name = resolved['active_terms_col']
            if active_col_name:
                ex['active_terms'] = to_num_series(ex[active_col_name])
                active_mode = 'column'
                active_status = 'ok'
            else:
                ex['active_terms'] = ex['term_cnt']
                active_mode = 'term_cnt'
                active_status = 'fallback_term_cnt'
                print(f'{month}: нет колонки активных терминалов, активность = Кол-во терминалов > 0')
            audit.append({
                'report_month': month,
                'field': 'active_terms',
                'excel_column': active_col_name,
                'status': active_status,
            })
            audit.append({
                'report_month': month,
                'field': 'open',
                'excel_column': resolved['open_col'],
                'status': open_status,
            })
            audit.append({
                'report_month': month,
                'field': 'close',
                'excel_column': resolved['close_col'],
                'status': close_status,
            })

            file_month = pd.Series(month, index=ex.index)
            open_m = (
                to_month_label(ex[resolved['open_col']])
                if resolved['open_col'] else pd.Series(pd.NA, index=ex.index)
            )
            close_m = (
                to_month_label(ex[resolved['close_col']])
                if resolved['close_col'] else pd.Series(pd.NA, index=ex.index)
            )
            p_clients = client_and_flow(
                ex, 'agr_id_key', 'active_terms', file_month, open_m, close_m, active_mode,
            )
            if resolved['open_col'] is None:
                p_clients['inflow'] = np.nan
            if resolved['close_col'] is None:
                p_clients['outflow'] = np.nan
            p_sums = dedup_sums(ex, 'inn_key', 'agr_id_key')
            for field, col_key in [
                ('retl_cnt', 'retl_col'),
                ('term_cnt', 'term_col'),
                ('trx_sum', 'trx_sum_col'),
                ('commission_from_ops', 'comm_ops_col'),
                ('commission_monthly', 'comm_monthly_col'),
                ('int_component', 'int_component_col'),
                ('chod', 'chod_col'),
                ('aur', 'aur_col'),
                ('amortization', 'amortization_col'),
                ('fin_result', 'fin_result_col'),
            ]:
                if resolved[col_key] is None:
                    p_sums[field] = np.nan
            portal = metrics_from_parts(p_clients, p_sums)

    for key, _label, _scale in METRIC_ROWS:
        rows.append({
            'report_month': month,
            'metric': key,
            'db': db.get(key, np.nan),
            'portal': portal.get(key, np.nan),
            'db_status': db_status,
        })

long_df = pd.DataFrame(rows)
long_df['delta'] = long_df['db'] - long_df['portal']
audit_df = pd.DataFrame(audit)
print('long rows', len(long_df), 'audit rows', len(audit_df))


In [ ]:
def wide_from(values):
    """values: DataFrame с колонками report_month, metric, db, portal, delta."""
    pieces = []
    for month in MONTHS:
        label = MONTH_LABEL[month]
        part = values.loc[values['report_month'] == month, ['metric', 'db', 'portal', 'delta']].copy()
        part = part.rename(columns={'db': (label, 'ДБ'), 'portal': (label, 'Портал'), 'delta': (label, 'Δ')})
        part = part.set_index('metric')
        pieces.append(part)
    wide = pd.concat(pieces, axis=1)
    wide.columns = pd.MultiIndex.from_tuples(wide.columns)
    order = [k for k, _l, _s in METRIC_ROWS]
    labels = {k: lab for k, lab, _s in METRIC_ROWS}
    wide = wide.reindex(order)
    wide.index = [labels[k] for k in wide.index]
    wide.index.name = 'Показатель'
    return wide


wide_rub = wide_from(long_df)

scale = {k: s for k, _l, s in METRIC_ROWS}
compact_src = long_df.copy()
for key, kind in scale.items():
    mask = compact_src['metric'] == key
    if kind == 'bln':
        compact_src.loc[mask, ['db', 'portal', 'delta']] = compact_src.loc[mask, ['db', 'portal', 'delta']] / 1e9
    elif kind == 'mln':
        compact_src.loc[mask, ['db', 'portal', 'delta']] = compact_src.loc[mask, ['db', 'portal', 'delta']] / 1e6
wide_compact = wide_from(compact_src)

print('Сверка, компактный вид: оборот млрд, остальные деньги млн, % как проценты')
display(wide_compact)
print('Сверка, рубли и штуки')
display(wide_rub)

jan = long_df.loc[long_df['report_month'] == '2026-01'].set_index('metric')
if len(jan):
    total = jan.loc['clients_total', 'db']
    active = jan.loc['clients_active', 'db']
    inactive = jan.loc['clients_inactive', 'db']
    print('Январь ДБ: всего', total, '| активные', active, '| неактивные', inactive, '| сумма', active + inactive)
    if pd.notna(total) and abs((active + inactive) - total) > 0.5:
        print('Январь: активные + неактивные не равны всему')
    print('Январь приток ДБ/Портал', jan.loc['inflow', 'db'], jan.loc['inflow', 'portal'])
    print('Январь отток ДБ/Портал', jan.loc['outflow', 'db'], jan.loc['outflow', 'portal'])
    if pd.isna(jan.loc['inflow', 'portal']) or pd.isna(jan.loc['outflow', 'portal']):
        print('Январь: портальный приток или отток пустой — смотрите лист «колонки»')

pretty = long_df.copy()
pretty['metric'] = pretty['metric'].map({k: lab for k, lab, _s in METRIC_ROWS})
pretty['month'] = pretty['report_month'].map(MONTH_LABEL)

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    wide_rub.to_excel(writer, sheet_name='сверка_руб')
    wide_compact.to_excel(writer, sheet_name='сверка_компакт')
    pretty.to_excel(writer, sheet_name='помесячно', index=False)
    if len(audit_df):
        audit_df.to_excel(writer, sheet_name='колонки', index=False)

print('Saved:', OUT_XLSX)
